In [2]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
import matplotlib.pyplot as plt
import time

**Klasa bazowa _LinBase_ implementujacą metody wspólne klasyfikatorów liniowych.**

In [3]:
class LinBase(BaseEstimator, ClassifierMixin):
    def __init__(self, coef=None, intercept=None, class_labels=None):
        self.coef_ = coef
        self.intercept_ = intercept
        self.class_labels_ = class_labels

    def margin(self, X, y):
        m = y * (X.dot(self.coef_) + self.intercept_)
        return np.min(m), m

    def decision_function(self, X):
        a, b = self.margin(X, np.ones(X.shape[0]))
        return b

    def predict_proba(self, X):
        a, b = self.margin(X, np.ones((X.shape[0],)))
        return np.array([1 - 1 / (1 + np.exp(-b)), 1 / (1 + np.exp(-b))]).T

    def predict(self, X):
        results = np.sign(X.dot(self.coef_) + self.intercept_)
        results_mapped = self.class_labels_[1 * (results > 0)]
        return results_mapped

    def get_params(self, deep=True):
        pass

    def set_params(self, **parameters):
        for parameter, value in parameters.items():
            setattr(self, parameter, value)
        return self

    def __str__(self):
        return "{}[w={}, b={}]".format(self.__class__.__name__, self.coef_, self.intercept_)

**Klasa Preceptorn , dziedzicząca po klasie LinBase i implementująca algorytm uczenia preceptronu zgodnie z regułą perceptronu**

In [4]:
class Perceptron(LinBase):
    def __init__(self, break_time=1, **kwargs):
        super(Perceptron, self).__init__(**kwargs)
        self.break_time_ = break_time

    def fit(self, X, y):
        m, n = X.shape
        self.class_labels_ = np.unique(y)
        if self.coef_ is not None:
            w = self.coef_
        else:
            w = np.zeros(n)

        if self.intercept_ is not None:
            b = self.intercept_
        else:
            b = 0

        i = 0
        k = 0
        t1 = time.time()
        while(k < m):
            if y[i] * (w.dot(X[i, :]) + b) <= 0:
                w = w + X[i, :] * y[i]
                b = b + y[i]
                k = 0
            else:
                k += 1
            i = (i + 1) % m
            t2 = time.time()
            if t2 - t1 > self.break_time_:
                print("Przekroczony czas!")
                break

        self.coef_ = w
        self.intercept_ = b

**Klasa VotedPerceptorn (uśrednianie decyzji)**

In [7]:
class AveragePerceptron(LinBase):
    def __init__(self, break_time=10, k_max=1000, **kwargs):
        super(AveragePerceptron, self).__init__(**kwargs)
        self.break_time_ = break_time
        self.v_ = None
        self.v_bias_ = None
        self.k_max = k_max

    def fit(self, X, y):
        m, n = X.shape
        self.class_labels_ = np.unique(y)
        if self.coef_ is not None:
            w = self.coef_
        else:
            w = np.zeros(n)

        if self.intercept_ is not None:
            b = self.intercept_
        else:
            b = 0

        self.v_ = 0 * w
        self.v_bias_ = 0 * b
        i = 0
        k = 0
        c = 0
        t1 = time.time()
        ci = 0
        while(k < self.k_max):
            ci += 1
            if y[i] * (w.dot(X[i, :]) + b) <= 0:
                self.v_ += c * w
                self.v_bias_ += c * b
                w = w + X[i, :] * y[i]
                b = b + y[i]
                k += 1
                c = 1
            else:
                c += 1
            i = (i + 1) % m
            t2 = time.time()
            if t2 - t1 > self.break_time_:
                print("Przekroczony czas!")
                break
        self.v_ += c * w
        self.v_bias_ += c * b
        print(f'Time: {time.time() - t1}\n'
              f'Iterations: {ci}\n'
              f'Number of weight changes: {c}')
        self.coef_ = self.v_
        self.intercept_ = self.v_bias_

**AveragedPerceptron (uśrednianie wag)**